In [2]:
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

pd.set_option('display.max_colwidth', 150)

In [3]:
DATA_PATH = '../dataset/labeled_tickets.csv'
df = pd.read_csv(DATA_PATH)
df.shape

(249497, 10)

In [4]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,text_clean,word_count,category
0,20,115715,True,Tue Oct 31 22:03:34 +0000 2017,"@115714 whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never in the 4 years I've tried https...",19,NaN,"whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never in the 4 years I've tried",25,Account
1,23,115716,True,Tue Oct 31 22:16:05 +0000 2017,@Ask_Spectrum The correct way to do it is via an OCS Account Takeover and email Consent Form it does not need to be done in a local office,NaN,21.0,The correct way to do it is via an OCS Account Takeover and email Consent Form it does not need to be done in a local office,27,Account
2,29,115716,True,Tue Oct 31 22:01:35 +0000 2017,actually that's a broken link you sent me and incorrect information https://t.co/V4yfrHR8VI,28,NaN,actually that's a broken link you sent me and incorrect information,11,Technical
3,41,115721,True,Tue Oct 31 22:24:55 +0000 2017,@VerizonSupport What else can I provide? They refuse to help me because they cannot validate the account...,43,40.0,What else can I provide? They refuse to help me because they cannot validate the account...,16,Account
4,47,115721,True,Tue Oct 31 22:01:42 +0000 2017,@115722 Nobody can find my account or number. I walked out of a store with this. I've explained that they can find my acct via my devices serial #..,46,48.0,Nobody can find my account or number. I walked out of a store with this. I've explained that they can find my acct via my devices serial #..,28,Account


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 249497 entries, 0 to 249496
Data columns (total 10 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   tweet_id                 249497 non-null  int64  
 1   author_id                249497 non-null  int64  
 2   inbound                  249497 non-null  bool   
 3   created_at               249497 non-null  str    
 4   text                     249497 non-null  str    
 5   response_tweet_id        227310 non-null  str    
 6   in_response_to_tweet_id  86469 non-null   float64
 7   text_clean               249497 non-null  str    
 8   word_count               249497 non-null  int64  
 9   category                 249497 non-null  str    
dtypes: bool(1), float64(1), int64(3), str(5)
memory usage: 17.4 MB


In [6]:
## weak priority labelling

In [7]:
urgent_keywords = ['urgent', 'asap', 'immediately', 'right now', 'emergency',
                    'still not', 'worst', 'unacceptable', 'furious']

def get_priority(text):
    text_lower = text.lower()
    has_kw = any(kw in text_lower for kw in urgent_keywords)
    has_exclaim = bool(re.search(r'!{2,}', text))
    has_caps = bool(re.search(r'\b[A-Z]{4,}\b', text))

    if has_kw or has_exclaim or has_caps:
        return 'High'
    return 'Normal'

df['priority'] = df['text_clean'].apply(get_priority)
df['priority'].value_counts()

priority
Normal    217939
High       31558
Name: count, dtype: int64

In [8]:
# train tesst split TF-IDF

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'].values, 
    df['priority'].values,
    test_size=0.2, 
    random_state=42, 
    stratify=df['priority'].values
)

In [10]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [11]:
# Logistic Regression classifier
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_tfidf, y_train)

preds = clf.predict(X_test_tfidf)
accuracy_score(y_test, preds)

0.7644288577154309

In [12]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

        High       0.29      0.58      0.38      6312
      Normal       0.93      0.79      0.85     43588

    accuracy                           0.76     49900
   macro avg       0.61      0.68      0.62     49900
weighted avg       0.85      0.76      0.79     49900



In [13]:
# Create a DataFrame to compare actual vs predicted for the test set
test_results = pd.DataFrame({
    'Text': X_test,
    'Actual Priority': y_test,
    'Predicted Priority': preds
})

print("Sample Test Set Predictions")
display(test_results.head(10))

# Filter for misclassified rows to analyze where the model is struggling
misclassified = test_results[test_results['Actual Priority'] != test_results['Predicted Priority']]
print("\nSample Misclassified Tickets (Error Analysis)")
display(misclassified.head(10))

Sample Test Set Predictions


,Text,Actual Priority,Predicted Priority
0,The issue got worse. I was instructed to reset and ended up losing all my files,Normal,Normal
1,I’m tired of my Mac crashing. I just want my computer to last ONE day without crashing. #macos #HighSierra #NeverAgain,Normal,High
2,"got this scam email pretending to be you's. know it's not you, as I have never had an account. Real customers should watch out.",Normal,Normal
3,my pc crashed while doing an update now i have this,Normal,Normal
4,"Been a TMobile customer for 12+ years, this month I get triple-billed b/c of billing glitch. #helpmeJohn",Normal,High
5,Can you inform me as to how long it takes for someone to get back me and where I can I call someone. A person has used my cards details! This is n...,Normal,High
6,"I bought by My friend was of vacation in Orlando, and I gave to amazon delivery for him in the his hotel.",Normal,Normal
7,I just updated to 11.1.1 and now my email won’t work. Tried signing out and signing back in. Is there a bug fix coming?,Normal,High
8,there is problem with payment system in the app. It's not allowing me to revert to cash. Card option is not working either.,Normal,Normal
9,I'm trying to socialize this evening son WHERE IS MY PACKAGE,High,Normal



Sample Misclassified Tickets (Error Analysis)


,Text,Actual Priority,Predicted Priority
1,I’m tired of my Mac crashing. I just want my computer to last ONE day without crashing. #macos #HighSierra #NeverAgain,Normal,High
4,"Been a TMobile customer for 12+ years, this month I get triple-billed b/c of billing glitch. #helpmeJohn",Normal,High
5,Can you inform me as to how long it takes for someone to get back me and where I can I call someone. A person has used my cards details! This is n...,Normal,High
7,I just updated to 11.1.1 and now my email won’t work. Tried signing out and signing back in. Is there a bug fix coming?,Normal,High
9,I'm trying to socialize this evening son WHERE IS MY PACKAGE,High,Normal
12,i want to cancel the cable tv portion of my package and upgrade to unlimited internet for the extra $50 a month,Normal,High
14,WHY DID MY CONTACTS JUST RANDOMLY DELETE OUT OF MY IPHONE? AND HOW DO I FIX IT?????????,High,Normal
22,"hello you have refunded me cheque estar bluetooth smart watch order id- 408-0814305-1670723,please send cheque on the name of my mother Kurakula.C...",Normal,High
25,Now I will unable to use my airtel data becoz it might be charged by airtel by saying that you hv used ur all data ne u hv to pay 4 it.,Normal,High
29,Good luck. British airways has really gone down hill. Planes are broken and dirty and we are supposed to accept this!,Normal,High


In [14]:
# n_jobs=-1 uses all cores, capped depth + fewer trees so this doesn't take forever
clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train_tfidf, y_train)

preds = clf.predict(X_test_tfidf)
accuracy_score(y_test, preds)

0.8131663326653307

In [15]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

        High       0.33      0.47      0.39      6312
      Normal       0.92      0.86      0.89     43588

    accuracy                           0.81     49900
   macro avg       0.62      0.67      0.64     49900
weighted avg       0.84      0.81      0.83     49900



In [16]:
# Create a DataFrame to compare actual vs predicted for the test set
test_results = pd.DataFrame({
    'Text': X_test,
    'Actual Priority': y_test,
    'Predicted Priority': preds
})

print("Sample Test Set Predictions")
display(test_results.head(10))

# Filter for misclassified rows to analyze where the model is struggling
misclassified = test_results[test_results['Actual Priority'] != test_results['Predicted Priority']]
print("\nSample Misclassified Tickets (Error Analysis)")
display(misclassified.head(10))

Sample Test Set Predictions


,Text,Actual Priority,Predicted Priority
0,The issue got worse. I was instructed to reset and ended up losing all my files,Normal,Normal
1,I’m tired of my Mac crashing. I just want my computer to last ONE day without crashing. #macos #HighSierra #NeverAgain,Normal,High
2,"got this scam email pretending to be you's. know it's not you, as I have never had an account. Real customers should watch out.",Normal,Normal
3,my pc crashed while doing an update now i have this,Normal,Normal
4,"Been a TMobile customer for 12+ years, this month I get triple-billed b/c of billing glitch. #helpmeJohn",Normal,High
5,Can you inform me as to how long it takes for someone to get back me and where I can I call someone. A person has used my cards details! This is n...,Normal,Normal
6,"I bought by My friend was of vacation in Orlando, and I gave to amazon delivery for him in the his hotel.",Normal,Normal
7,I just updated to 11.1.1 and now my email won’t work. Tried signing out and signing back in. Is there a bug fix coming?,Normal,Normal
8,there is problem with payment system in the app. It's not allowing me to revert to cash. Card option is not working either.,Normal,Normal
9,I'm trying to socialize this evening son WHERE IS MY PACKAGE,High,Normal



Sample Misclassified Tickets (Error Analysis)


,Text,Actual Priority,Predicted Priority
1,I’m tired of my Mac crashing. I just want my computer to last ONE day without crashing. #macos #HighSierra #NeverAgain,Normal,High
4,"Been a TMobile customer for 12+ years, this month I get triple-billed b/c of billing glitch. #helpmeJohn",Normal,High
9,I'm trying to socialize this evening son WHERE IS MY PACKAGE,High,Normal
14,WHY DID MY CONTACTS JUST RANDOMLY DELETE OUT OF MY IPHONE? AND HOW DO I FIX IT?????????,High,Normal
27,how do I get a refund for a service appointment one of your technicians bailed on?,Normal,High
31,Thankyou - that worked. Very very weird that they were spontaneously deleted and two playlists with... strange.... titles were created. Am absolut...,Normal,High
33,"Oh, you know, just ANOTHER delayed package. This is a regular thing now. Good thing we’re done paying for “Prime” shipping at the end of the month...",High,Normal
37,. Gotta fix the super EE glitch of getting thrown off the map. So tired of losing cause of that!,Normal,High
38,"All I expect is what I ordered &amp; paid for. Don't charge me for 100% of the items I ordered, tell me you shipped 100% of them &amp; when I rece...",Normal,High
43,I need for and the update to get it together! Spotify is not working properly since update! I pay for this.,Normal,High


In [17]:
# RF improves overall performance but trades off recall on the priority class that matters most, so the choice depends on whether false negatives or false positives are more costly for this use case

In [32]:
import pickle

with open('saved_models/priority_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

with open('saved_models/priority_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

In [19]:
# category stays as the weak label (same one used to train/test the classifier)
# priority is now the model's predictions on the FULL dataset, not just weak labels
full_tfidf = tfidf.transform(df['text_clean'])
df['priority'] = clf.predict(full_tfidf)

df.to_csv('../dataset/labeled_tickets_with_priority.csv', index=False)
print(df.shape)
df[['text_clean', 'category', 'priority']].head()

(249497, 11)


,text_clean,category,priority
0,"whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never in the 4 years I've tried",Account,Normal
1,The correct way to do it is via an OCS Account Takeover and email Consent Form it does not need to be done in a local office,Account,Normal
2,actually that's a broken link you sent me and incorrect information,Technical,Normal
3,What else can I provide? They refuse to help me because they cannot validate the account...,Account,Normal
4,Nobody can find my account or number. I walked out of a store with this. I've explained that they can find my acct via my devices serial #..,Account,Normal
